# 🐺 GökTürk-2.5B-Thinking — Colab T4 Eğitimi
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a23521745-hub/GokTurk-2.5B-Thinking/blob/main/train/GokTurk_Colab_T4.ipynb)

**Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin, sonra **Tümünü çalıştır**.

5 aşamalı format: `<think>` → `<plan>` → `<search_query>` → `<verify>` → `<output>`

In [ ]:
#@title ⚙️ Ayarlar
MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" #@param ["unsloth/Qwen2.5-3B-Instruct-bnb-4bit", "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"]
EPOCHS = 2 #@param {type:"number"}
MAX_STEPS = -1 #@param {type:"integer"}
SYNTH_N = 3000 #@param {type:"integer"}
GGUF = "" #@param ["", "q4_k_m", "q4_k_m q8_0"]
HF_REPO = "" #@param {type:"string"}

In [ ]:
#@title 📦 Kurulum + repo
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth
!git clone -q https://github.com/a23521745-hub/GokTurk-2.5B-Thinking.git || (cd GokTurk-2.5B-Thinking && git pull -q)
%cd GokTurk-2.5B-Thinking

In [ ]:
#@title 📚 (İsteğe bağlı) Kendi JSONL verinizi yükleyin — atlanırsa sentetik veri üretilir
import glob
DATA = sorted(glob.glob('data/*.jsonl'))
DATA = [d for d in DATA if not d.endswith('template.jsonl')] or ['data/gokturk_synth.jsonl']
print(DATA)

In [ ]:
#@title 🚀 Eğit
import shlex
cmd = f"python train/train_unsloth.py --model {MODEL} --epochs {EPOCHS} --max-steps {MAX_STEPS} --synth-n {SYNTH_N} --data {' '.join(DATA)}"
if GGUF: cmd += f" --gguf {GGUF}"
if HF_REPO: cmd += f" --push-to-hub {HF_REPO}"
print(cmd)
!{cmd}

In [ ]:
#@title 💾 Çıktıları Drive'a kopyala (isteğe bağlı)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/gokturk && cp -r outputs/gokturk-2.5b-thinking/lora /content/drive/MyDrive/gokturk/
!ls outputs/gokturk-2.5b-thinking/gguf 2>/dev/null && cp -r outputs/gokturk-2.5b-thinking/gguf /content/drive/MyDrive/gokturk/ || true